# StopTheCap - Kaggle resource measurement (private project)

Measures the GPU actually assigned and what Kaggle exposes. It does **not** train E1.

**Privacy:** the project arrives as a **private Kaggle Dataset**. Nothing is published, no repository
is cloned, no credentials are requested.

## Before you run: three settings checks

1. **Input panel (right side)** - your private dataset must be attached and show as e.g.
   `stopthecap`. The path will be `/kaggle/input/<slug>/`.
2. **Settings -> Internet: ON** - required for `pip install`.
3. **Settings -> Accelerator: GPU** - choosing it restarts the session once. The GPU model is not
   guaranteed; the notebook records whatever you get.

Note: do **not** create a copy of the notebook from the "File" menu after attaching data - use this
notebook directly so the attached input and settings stay with it.

## 1. Configure

In [ ]:
REPO_DIR       = "/kaggle/working/StopTheCap"
REPORT_DIR     = "/kaggle/working/StopTheCap_reports"
CHECKPOINT_DIR = REPORT_DIR + "/ckpt_probe"
INPUT_ROOT     = "/kaggle/input"

print("repo dir :", REPO_DIR)
print("report   :", REPORT_DIR)
print("inputs   :", INPUT_ROOT)

## 2. Load the project

The dataset may present the project in either form, and the slug varies by account, so the loader
**auto-discovers** it instead of trusting a hard-coded path:

- `StopTheCap.zip` somewhere under `/kaggle/input/<slug>/` (extracted here), or
- the project already unpacked (`.../<slug>/ml/cloud/probe.py`).

If neither is found, it prints exactly what *is* under `/kaggle/input` so the problem is visible.

In [ ]:
# If the config cell was skipped (or the kernel restarted), fall back to the documented defaults.
if "REPO_DIR" not in globals():
    REPO_DIR = "/kaggle/working/StopTheCap"
    INPUT_ROOT = "/kaggle/input"
    print("NOTE: the config cell had not run; using default paths.")

import os, shutil, sys, zipfile
from pathlib import Path

REQUIRED = ["ml/__init__.py", "ml/cloud/__init__.py", "ml/cloud/probe.py"]
EXTRACT_TMP = "/kaggle/working/_stc_extract"

def _fail(msg):
    raise RuntimeError(msg)

def _looks_like_project(p: Path) -> bool:
    return (p / "ml" / "cloud" / "probe.py").is_file()

def find_source(input_root: str = "/kaggle/input"):
    root = Path(input_root)
    if not root.is_dir():
        _fail("No " + input_root + " directory. Attach your private dataset via the Input panel.")
    entries = sorted(p for p in root.iterdir() if p.is_dir())
    for d in entries:
        zips = [z for z in d.rglob("*.zip") if "stopthecap" in z.name.lower()]
        if zips:
            return ("zip", zips[0])
    for d in entries:
        if _looks_like_project(d):
            return ("dir", d)
        if _looks_like_project(d / "StopTheCap"):
            return ("dir", d / "StopTheCap")
    listing = ", ".join(p.name for p in entries) or "(empty)"
    _fail("Could not find the project under " + input_root + ".\n"
          "Found: " + listing + "\n"
          "Attach a private dataset containing StopTheCap.zip (or the unpacked project).")

def extract_zip(src, dest):
    dest = Path(dest)
    tmp = Path(EXTRACT_TMP)
    for p in (dest, tmp):
        if p.exists():
            shutil.rmtree(p)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(src) as zf:
        names = zf.namelist()
        if not names:
            _fail("The ZIP is empty.")
        print("zip entries:", len(names))
        for n in names[:10]:
            print("   ", n)
        zf.extractall(tmp)
    entries = [p for p in tmp.iterdir() if p.name != "__MACOSX"]
    root = entries[0] if len(entries) == 1 and entries[0].is_dir() else tmp
    shutil.move(str(root), str(dest))
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    return dest

def copy_dir(src, dest):
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
    return dest

kind, source = find_source(INPUT_ROOT)
print("found:", kind, "->", source)

REPO = extract_zip(source, REPO_DIR) if kind == "zip" else copy_dir(source, REPO_DIR)
REPO = Path(REPO).resolve()

missing = [f for f in REQUIRED if not (REPO / f).is_file()]
if missing:
    _fail("Project structure looks wrong; missing: " + ", ".join(missing) +
          "\nThe dataset must contain the repository root (the folder holding ml/, datasets/, ...).")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print()
print("project root :", REPO)
print("top level    :", sorted(p.name for p in REPO.iterdir())[:14])

## 3. Dependencies

In [ ]:
# torch ships with the Kaggle image; only transformers is added (the benchmark needs it).
%pip install -q "transformers>=4.40"

import torch, transformers
print("torch       :", torch.__version__)
print("cuda build  :", torch.version.cuda)
print("transformers:", transformers.__version__)
print("cuda present:", torch.cuda.is_available())

## 4. Run the measurement

In [ ]:
# 30 optimiser steps on synthetic tensors. NOT E1 training: no dataset is read.
from pathlib import Path
from ml.cloud.probe import run_probe

REPORT_PATH = REPO / "reports/cloud/kaggle_measure.json"
res = run_probe(
    out=REPORT_PATH,
    benchmark_model="distilbert-base-cased",
    steps=30,
    checkpoint_dir=Path(CHECKPOINT_DIR),
)
print("wrote", REPORT_PATH)
print("benchmark:", res.get("benchmark"))

## 4b. Two-GPU check (T4 x2 runtimes only)

Runs the same benchmark as a **separate process per card** and compares summed throughput against
one card. This measures whether the second GPU actually helps this workload, rather than assuming it.

Two independent processes - not DDP - because the encoder fits on one card and Kaggle gives only
2 vCPUs, so the dataloader is the more likely bottleneck.


In [ ]:
# Takes about a minute. On a single-GPU runtime this fails gracefully and can be skipped.
!python -m ml.cloud.multigpu_bench --devices 0,1 --steps 20 --out reports/cloud/multigpu_measure.json

import json, pathlib
p = pathlib.Path("reports/cloud/multigpu_measure.json")
if p.is_file():
    print(json.dumps(json.loads(p.read_text())["summary"], indent=2))
else:
    print("No 2-GPU result (single-GPU runtime?). Safe to skip.")


## 5. Persist the report

In [ ]:
import shutil
from pathlib import Path

Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
final = Path(REPORT_DIR) / "kaggle_measure.json"
shutil.copy2(REPORT_PATH, final)
print("report:", final)
print("bytes :", final.stat().st_size)
print()
print("To keep it: File -> Save Version -> Save & Run All.")
print("Then open the notebook's Output tab and download kaggle_measure.json.")

## 6. Report

In [ ]:
import json
from pathlib import Path

d = json.loads(Path(REPORT_PATH).read_text(encoding="utf-8"))

print("host       :", d["platform"]["host"])
print("cuda       :", d["gpu"]["cuda_available"])
for dev in d["gpu"].get("devices", []):
    print("GPU        :", dev["name"], "|", dev["total_vram_gb"], "GB")
print("ram        :", d["memory"])
print("disk       :", d["disk"])
print("benchmark  :", d.get("benchmark"))
print("checkpoint :", d.get("checkpoint"))
print()
print(json.dumps(d, indent=2))

## Two values to record by hand

1. **Remaining GPU hours this week** - Kaggle shows an accelerator quota in Settings or the account
   page. Copy the figure.
2. **Session limit** - Kaggle documents 12 h for CPU/GPU notebooks; confirm what your account shows.

## What this notebook does NOT do

No E1 training, no dataset download, no model checkpoint, no change to split manifests or the frozen
threshold, no publishing.